In [1]:
import os
import json
import torch
from pathlib import Path
from datasets import load_dataset
from transformers import (
    LlamaTokenizer,
    LlamaForCausalLM,
    GenerationConfig,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    default_data_collator,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType

In [2]:
DATA_PATH = "E:\Work\ML\Config Generation Server\datasets\draft_1.jsonl"
OUTPUT_DIR = "C:\\Users\\User\\Documents\\Work\\Config Generation Server\\Saved_fine_tuned_model\\llama-3.2-3b-instruct\\saved_model"
INSTRUCTION_PREFIX = (
    "Below is a Config description in natural language. "
    "Write the exact JSON configuration (conforming to the schema) that fulfills the user's requirements.\n\n"
)
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

In [3]:
MAX_SEQ_LENGTH = 512

LORA_R = 4             
LORA_ALPHA = 16      
LORA_DROPOUT = 0.05 
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj"
] 

In [4]:
# Training hyperparameters
NUM_EPOCHS = 4         
LR = 2e-5                 # learning rate
PER_DEVICE_BATCH_SIZE = 4 # batch size per GPU (we use accumulation to simulate larger batch)
ACCUMULATION_STEPS = 4    # effective batch size = 1×8 = 8
FP16 = True               # use mixed precision
GRADIENT_CHECKPOINTING = True  # save VRAM by checkpointing
SAVE_TOTAL_LIMIT = 2      # keep only the 2 most recent checkpointse vh

In [5]:


dataset = load_dataset("json", data_files=DATA_PATH, split="train")

print(dataset[0]["input"])
print(json.dumps(dataset[0]["output"], ensure_ascii=False))


Generating train split: 0 examples [00:00, ? examples/s]

I want a elegant Config with a digital clock at top and calendar as a box.
{"Config": {"Background": {"color": "red", "image": null}, "Widget": {"color": "green", "Item": [{"name": "box_calendar", "type": "box", "position": "bottom"}]}, "Clock": {"type": "digital", "color": "green", "position": "top"}, "Dial": null}}


In [6]:
def format_example(example):
    user_desc = example["input"].strip()

    json_config = json.dumps(example["output"], ensure_ascii=False)
    # Build a single text: prompt -> output
    # prompt_text = f"{INSTRUCTION_PREFIX}user_description: \"{user_desc}\"\n\ngenerate_config:\n{json_config}"
    prompt_text = (
        f"{INSTRUCTION_PREFIX}"
        f"user_description: \"{user_desc}\"\n\n"
        f"generate_config:\n"
        f"{json_config}"
    )
    return {"prompt_text": prompt_text}


formatted = dataset.map(
    format_example,
    remove_columns=dataset.column_names, 
    # num_proc=4,                      
)
split = formatted.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
eval_ds  = split["test"]

Map:   0%|          | 0/22599 [00:00<?, ? examples/s]

In [7]:
print(f"▶ Dataset: {len(train_ds)} train examples, {len(eval_ds)} validation examples")

▶ Dataset: 20339 train examples, 2260 validation examples


In [8]:
eval_ds[0]["prompt_text"]  # check a sample

'Below is a Config description in natural language. Write the exact JSON configuration (conforming to the schema) that fulfills the user\'s requirements.\n\nuser_description: "Make a football pitch layout, circle for goals scored, box for possession percentage, and digital clock near the top. Camera whom during. Able morning leader."\n\ngenerate_config:\n{"Config": {"Background": {"color": "aqua", "image": null}, "Widget": {"color": "coral", "Item": [{"name": "edge_uv_index", "type": "edge", "position": "top_right"}, {"name": "edge_worldclock", "type": "edge", "position": "bottom_right"}, {"name": "arc_weekly_activity", "type": "arc", "position": "top"}]}, "Clock": {"type": "digital", "color": "coral", "position": "middle"}, "Dial": null}}'

In [9]:
# 

In [10]:
# import transformers
# import torch

# model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# pipeline = transformers.pipeline(
#     "text-generation",
#     model=model_id,
#     model_kwargs={"torch_dtype": torch.bfloat16},
#     device_map="auto",
# )

# messages = [
#     {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
#     {"role": "user", "content": "Who are you?"},
# ]

# outputs = pipeline(
#     messages,
#     max_new_tokens=256,
# )
# print(outputs[0]["generated_text"][-1])


In [11]:
# from huggingface_hub import snapshot_download
# # If you want to download to a local cache folder explicitly:
# local_dir = snapshot_download("meta-llama/Meta-Llama-3.1-8B-Instruct")
# print("Files downloaded to:", local_dir)

In [12]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(example):

    tokenized = tokenizer(
        example["prompt_text"],
        max_length=MAX_SEQ_LENGTH,
        truncation=True,
        padding="max_length",
    )

    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized


tokenized_train = train_ds.map(
    tokenize_function,
    batched=True,
    remove_columns=train_ds.column_names,
    # num_proc=4,
)
tokenized_eval = eval_ds.map(
    tokenize_function,
    batched=True,
    remove_columns=eval_ds.column_names,
    # num_proc=4,
)

Map:   0%|          | 0/20339 [00:00<?, ? examples/s]

Map:   0%|          | 0/2260 [00:00<?, ? examples/s]

In [13]:
# from torch.utils.data import DataLoader

# # 4) DataLoaders
# train_loader = DataLoader(tokenized_train, batch_size=PER_DEVICE_BATCH_SIZE, shuffle=True, collate_fn=lambda x: tokenizer.pad(x, return_tensors="pt"))
# eval_loader  = DataLoader(tokenized_eval,  batch_size=PER_DEVICE_BATCH_SIZE, collate_fn=lambda x: tokenizer.pad(x, return_tensors="pt"))

In [14]:

model = LlamaForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map=None            # auto‐place model shards on GPU
)

model.eval()
model = model.to("cuda")



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [15]:

if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()


lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
)


model = get_peft_model(model, lora_config)




In [16]:
# from accelerate import Accelerator

# # Initialize Accelerator
# accelerator = Accelerator(
#     gradient_accumulation_steps=ACCUMULATION_STEPS,
#     mixed_precision="fp16" if FP16 else "no",
# )

# model, train_loader, eval_loader = accelerator.prepare(model, train_loader, eval_loader)

# # Optimizer
# optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

In [17]:
# # Training Loop
# for epoch in range(NUM_EPOCHS):
#     model.train()
#     total_loss = 0.0
#     for step, batch in enumerate(train_loader):
#         outputs = model(**batch)
#         loss = outputs.loss
#         accelerator.backward(loss)
#         if (step + 1) % ACCUMULATION_STEPS == 0:
#             optimizer.step()
#             optimizer.zero_grad()
#         total_loss += loss.item()
#     avg_train_loss = total_loss / len(train_loader)
#     print(f"Epoch {epoch+1} train_loss: {avg_train_loss:.4f}")

#     # Evaluation
#     model.eval()
#     eval_loss = 0.0
#     for batch in eval_loader:
#         with torch.no_grad():
#             outputs = model(**batch)
#         eval_loss += outputs.loss.item()
#     avg_eval_loss = eval_loss / len(eval_loader)
#     print(f"Epoch {epoch+1} eval_loss: {avg_eval_loss:.4f}")

#     #  Save checkpoint
#     ckpt_dir = os.path.join(OUTPUT_DIR, f"checkpoint-epoch{epoch+1}")
#     accelerator.save_state(ckpt_dir)
#     print(f" Checkpoint saved to {ckpt_dir}")

In [18]:
# # Final Save
# accelerator.wait_for_everyone()
# unwrapped_model = accelerator.unwrap_model(model)
# unwrapped_model.save_pretrained(OUTPUT_DIR)
# tokenizer.save_pretrained(OUTPUT_DIR)
# print("Fine-tuning complete, final model & tokenizer saved to", OUTPUT_DIR)

In [19]:

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=ACCUMULATION_STEPS,
    learning_rate=LR,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=SAVE_TOTAL_LIMIT,
    fp16=FP16,
    remove_unused_columns=False,  
    report_to="none",      
)

data_collator =  DataCollatorForSeq2Seq(tokenizer, model=model)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

C:\Users\User\AppData\Local\Temp\ipykernel_18468\2126700052.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:

trainer.train()

In [ ]:

trainer.save_state()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

In [ ]:
saved_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

model = LlamaForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

In [ ]:
def generate_config_json(user_input: str) -> str:
    prompt = (
        f"{INSTRUCTION_PREFIX}"
        f"user_description: \"{user_input.strip()}\"\n\n"
        f"generate_config:\n"
    )
    # prompt = user_input
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        num_beams=4,
        early_stopping=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Strip off the prompt to return only the JSON
    return text#.replace(prompt, "").strip()

# Test
sample = (
    "ANIMATED  ANALOG config with a black background, "
)
print(generate_config_json(sample))